# DEMO dei database persistenti

Seleziona **database e dimensione** in `config/demo.yaml`. Le esecuzioni sono lette da `benchmark.demo_repetitions` in `config/benchmark.yaml` (default 3). Avvia dall'host Jupyter e un solo DB; con Compose diretto arresta esplicitamente gli altri. Gli script PowerShell applicano anche il controllo sui container.

Questa DEMO usa esclusivamente destinazioni e riferimenti di campagne concluse: niente download, preprocessing, ingestione, reset, ANALYZE o creazione/rimozione di indici. Se mancano dati o indici, si ferma. Non usare i notebook FULL per la dimostrazione e non riprendere le campagne approvate.

In [ ]:
from pathlib import Path
import sys
ROOT=next(p for p in (Path.cwd(), *Path.cwd().parents)
          if (p/'src').is_dir() and (p/'config').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from IPython.display import display, HTML, JSON, SVG
from src.demo import load_demo_settings, demo_configuration, run_demo
from src.history import read_experiment, read_catalog, read_json
from src.presentation import html_table, query_descriptions, demo_rows, historical_charts
settings=load_demo_settings(ROOT/'config/demo.yaml')
from src.presentation import demo_comparison_html


In [ ]:
history=read_experiment(settings.campaign, settings.experiment.database,
                        settings.experiment.dataset_size, settings.experiment.results_dir)
config=demo_configuration(settings.experiment, history)
display(JSON({'modalita':'DEMO', 'campagna_storica':str(settings.campaign),
              'configurazione':config.manifest(),
              'versione_DB_storica':history['snapshot']['database_version'],
              'checksum_dataset':history['reference']['dataset_sha256']}))
display(HTML(html_table(query_descriptions(config),('query','operazione','parametri/tipi'))))

## Esecuzione leggera e controlli

Imposta `ESEGUI_DEMO=True` e lancia la cella. Una sola connessione per sessione; 0 warm-up e 3 esecuzioni reali di ciascuna query. Il timer condiviso con FULL include invio e acquisizione del risultato; confronto, digest e visualizzazione sono esterni.

Il controllo minimo verifica schema, indici attesi, conteggio esatto e campioni Q1. Ogni risultato Q1-Q4 viene confrontato con il riferimento CSV storico. Gli audit integrali NULL/BSON non vengono ripetuti: appartengono alla certificazione FULL. Anche tre Q4 sul dataset completo richiedono scansioni reali: per la prova piu rapida scegli 100K.

SQL usa una sessione read-only; Mongo usa le credenziali esistenti e una facciata senza API di scrittura. Questo protegge il percorso DEMO, non codice Python arbitrario scritto fuori da esso. Gli errori fermano la sessione, conservando raw e diagnostica in un nuovo run DEMO, senza ripristini automatici.

In [ ]:
ESEGUI_DEMO=False
DEMO_DIR=None
if ESEGUI_DEMO:
    DEMO_DIR=run_demo(settings)
    print('Nuovo run esclusivamente DEMO:',DEMO_DIR)
else:
    print('Nessuna query avviata. Configurazione e storici sono consultabili offline.')

## Risultati nuovi e storici, sempre etichettati

I tempi DEMO non si aggiungono alle statistiche FULL. Q2/Q3 attuali usano i secondari gia presenti; la baseline senza secondari viene mostrata soltanto dallo storico. Per Q1/Q4 lo storico e stato misurato con sola PK, mentre la DEMO conserva gli indici attuali: gli scenari sono etichettati, senza confronti percentuali impliciti.
La tabella LIVE affianca atteso, ottenuto e correttezza per ogni ripetizione. Espandere i dettagli per confrontare tutti i campi Q1 e le medie Q4; la correttezza usa la stessa tolleranza del laboratorio. Senza abilitazione vengono mostrati soltanto attesi e storici, mai risultati LIVE inventati.


In [ ]:
columns=('mode','database','dataset_size','query_id','index_state','valid_repetitions','mean_ms','source_run')
print('STORICO FULL APPROVATO')
display(HTML(html_table([r for r in history['summaries'] if r['event']=='query'],columns)))
if DEMO_DIR is not None:
    print('LIVE DEMO: nuove misure dimostrative, separate dallo storico FULL')
    display(HTML(html_table(demo_rows(DEMO_DIR),columns)))
    print('LIVE DEMO: confronto atteso / ottenuto / correttezza')
    display(HTML(demo_comparison_html(history['reference'], read_json(DEMO_DIR/'query_results.json'))))
    display(JSON(read_json(DEMO_DIR/'readiness.json')))
else:
    print('LIVE DEMO non eseguita: nessun risultato ottenuto nella sessione corrente.')
    display(JSON({'ATTESI DAL RIFERIMENTO FULL': {q: history['reference'][q] for q in ('Q2','Q3','Q4')}, 'ID Q1 attesi': config.q1_ids}))


## Piani disponibili: provenienza storica FULL

I piani seguenti sono quelli salvati durante la campagna selezionata, prima e dopo gli indici. Non vengono ricalcolati dalla DEMO e non certificano il planner corrente. L'eventuale presenza di avvisi resta visibile. Consultare il notebook 06 per grafici e confronto delle tre dimensioni.

In [ ]:
display(JSON({'origine':'storico FULL, nessun nuovo EXPLAIN', 'piani':history['plans']}))